# Graph Construction Demo

This notebook demonstrates how brain MRI supervoxels are converted into a graph
representation suitable for Graph Attention Networks.

## Method Overview

**Eq. 7 (Edge construction):** Two supervoxel nodes are connected by an edge if their
3D regions are spatially adjacent (share boundary voxels). Edge weights encode the
similarity between connected supervoxels based on:

- Intensity statistics across all modalities (mean, std, percentiles)
- Shared boundary length (number of adjacent voxel pairs)
- Spatial proximity (centroid distance)

**Node features** are computed per supervoxel:
- Per-modality statistics: mean, std, 25th/50th/75th percentiles (5 features x 4 modalities = 20)
- Spatial features: centroid coordinates (x, y, z) normalized to [0, 1] (3 features)
- Volumetric features: supervoxel size, surface area (2 features)
- Atlas region: one-hot encoded anatomical label (96 features)
- **Total: 121 features per node**

In [ ]:
%matplotlib inline

import sys
sys.path.insert(0, "../src")

import numpy as np
import matplotlib.pyplot as plt

from stroke_gat.config import Config
from stroke_gat.data.service import DataService
from stroke_gat.graph.builder import GraphBuilder
from stroke_gat.visualization.graph_3d import (
    plot_graph_3d_interactive,
    plot_graph_2d_projection,
)

print("Imports successful.")

In [ ]:
# Load configuration and build graph for one subject
cfg = Config.from_yaml("../configs/default.yaml")
data_service = DataService(cfg)

subjects = data_service.discover_subjects()
subject_id = subjects[0]
print(f"Building graph for subject: {subject_id}")

# GraphBuilder handles the full pipeline:
# load volumes -> SLIC -> feature extraction -> edge construction
builder = GraphBuilder(cfg, data_service)
graph = builder.build(subject_id)

print(f"\nGraph construction complete.")

In [ ]:
# Print detailed graph statistics
print(f"Graph Statistics for Subject {subject_id}")
print("=" * 50)
print(f"Number of nodes:     {graph.num_nodes:>8,}")
print(f"Number of edges:     {graph.num_edges:>8,}")
print(f"Node feature dim:    {graph.x.shape[1]:>8}")
print(f"Average degree:      {graph.num_edges / graph.num_nodes:>8.1f}")

# Edge weight statistics
if hasattr(graph, 'edge_weight') and graph.edge_weight is not None:
    ew = graph.edge_weight.numpy()
    print(f"\nEdge Weight Statistics:")
    print(f"  Min:    {ew.min():.4f}")
    print(f"  Max:    {ew.max():.4f}")
    print(f"  Mean:   {ew.mean():.4f}")
    print(f"  Median: {np.median(ew):.4f}")
    print(f"  Std:    {ew.std():.4f}")

# Node label distribution
if hasattr(graph, 'y') and graph.y is not None:
    labels = graph.y.numpy()
    print(f"\nNode Label Distribution:")
    for label, name in enumerate(["Normal", "Penumbra", "Core"]):
        count = np.sum(labels == label)
        pct = 100.0 * count / len(labels)
        print(f"  {name:>10s}: {count:>6,} nodes ({pct:.1f}%)")

In [ ]:
# Visualize node feature distributions
features = graph.x.numpy()

# Show the first 20 features (intensity statistics: 5 per modality x 4 modalities)
modality_names = ["T1", "FLAIR", "ADC", "TRACE"]
stat_names = ["Mean", "Std", "P25", "P50", "P75"]

fig, axes = plt.subplots(4, 5, figsize=(20, 12))
for i, mod in enumerate(modality_names):
    for j, stat in enumerate(stat_names):
        feat_idx = i * 5 + j
        ax = axes[i, j]
        ax.hist(features[:, feat_idx], bins=40, color="steelblue",
                edgecolor="white", alpha=0.8)
        ax.set_title(f"{mod} - {stat}", fontsize=10)
        ax.tick_params(labelsize=8)

fig.suptitle("Node Feature Distributions (Intensity Statistics)",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Edge weight distribution
if hasattr(graph, 'edge_weight') and graph.edge_weight is not None:
    ew = graph.edge_weight.numpy()

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].hist(ew, bins=60, color="coral", edgecolor="white", alpha=0.85)
    axes[0].set_xlabel("Edge Weight", fontsize=12)
    axes[0].set_ylabel("Count", fontsize=12)
    axes[0].set_title("Edge Weight Distribution", fontsize=13)

    # Degree distribution
    edge_index = graph.edge_index.numpy()
    degrees = np.bincount(edge_index[0], minlength=graph.num_nodes)

    axes[1].hist(degrees, bins=30, color="mediumpurple", edgecolor="white", alpha=0.85)
    axes[1].set_xlabel("Node Degree", fontsize=12)
    axes[1].set_ylabel("Count", fontsize=12)
    axes[1].set_title(f"Degree Distribution (mean={degrees.mean():.1f})", fontsize=13)

    plt.tight_layout()
    plt.show()

In [ ]:
# Interactive 3D graph visualization
# Nodes are colored by their ground-truth label; edges shown as lines
fig = plot_graph_3d_interactive(
    graph=graph,
    color_by="label",
    title=f"Brain Graph - Subject {subject_id}",
    node_size=3,
    edge_alpha=0.1,
)
fig.show()

In [ ]:
# 2D projection of the graph using t-SNE on node features
fig = plot_graph_2d_projection(
    graph=graph,
    method="tsne",
    color_by="label",
    title=f"t-SNE Projection of Node Features - Subject {subject_id}",
)
plt.show()

## Notes on Graph Properties

**Connectivity:**
- The graph is typically connected (or nearly so) since supervoxels tile the brain volume.
- Average degree reflects the 3D adjacency structure; interior supervoxels have more
  neighbors than those at the brain surface or atlas region boundaries.

**Feature Space:**
- The 121-dimensional node features capture multi-scale information: local intensity
  (per-modality statistics), spatial location (normalized coordinates), and anatomical
  context (atlas one-hot encoding).
- The t-SNE projection often shows some separation between lesion and normal nodes,
  but with significant overlap, motivating the use of message-passing (GAT) to
  leverage neighborhood context.

Next: See `04_training_demo.ipynb` for training the Graph Attention Network.